# S7 (last experiment) — physics-based acquisition layer
No training. Rules: `S7_preregistration.txt` (+ Addendum 1), fixed before any S7 code ran.

The diffusion model draws a clean image; a real mammogram is that image **plus acquisition noise**.
S7 fits the Poisson–Gaussian noise-level function σ²(I) = a·I + b on the **real train-fit images only** and adds that
noise to the generated baseline images (`v_base`, epoch 40, because S6 had no winner).

* **Validation:** `v_base` vs `v_base + noise`, reference = train-fit images, 95 % CI.
* **Test (only if validation PASSES, run once):** `t_base` vs `t_base + noise`, fresh noise seed, reference = TEST images, **97.5 % CI**.
* PASS = KID-mammo ≤ 0.90× with paired CI < 0, KID-DINOv2 ≤ 1.00×, recall ≥ baseline − 0.05 (same as S6).

**How to run:** Add Input → **clean-data-baseline-retraining**, **S2** and **S6 (final)** outputs · GPU T4 ×2 · Internet ON ·
Save & Run All (Commit) · stop the session. **≈ 30–40 min.** Send back `S7_results.zip`.

In [ ]:
# ===================== CELL 1 — CONFIG =====================
BUDGET_H = 3.0
VAL_SEED, TEST_SEED = 7007, 9707      # noise seeds (pre-registered, never used before)
TEST_ALPHA = 0.025                    # 97.5 % CI at the test stage (2nd confirmation in the project)
RUN_SMOKE = True
import os, time
os.environ["PA_OUT"] = "/kaggle/working/S7"
os.makedirs(os.environ["PA_OUT"], exist_ok=True)
if not os.path.exists("/tmp/pg_session_start"):
    open("/tmp/pg_session_start", "w").write(str(time.time()))
def used_h():
    return (time.time() - float(open("/tmp/pg_session_start").read())) / 3600
print("S7 acquisition layer | budget", BUDGET_H, "h | output", os.environ["PA_OUT"])

In [ ]:
# ===================== CELL 2 — setup (needs Internet ON) =====================
import os, re, sys, subprocess
os.chdir("/kaggle/working")
if not os.path.isdir("/kaggle/working/U24_synthesis/.git"):
    subprocess.run("git clone -q https://github.com/Its-Me-Sk/U24_synthesis.git", shell=True, check=True)
if not os.path.exists("/tmp/pip_done"):
    # Kaggle runs Python 3.13: install the repo requirements once, never force-reinstall numpy / pandas
    subprocess.run("cd /kaggle/working/U24_synthesis && grep -v xformers requirements.txt > /tmp/req.txt && "
                   "pip install -q -r /tmp/req.txt", shell=True, check=True)
    subprocess.run("pip install -q --no-deps torch-fidelity", shell=True, check=True)
    open("/tmp/pip_done", "w").write("ok")
probe = ("import sys, numpy, pandas, scipy, torch, torchmetrics, monai, generative, PIL; "
         "print('python', sys.version.split()[0], '| numpy', numpy.__version__, '| pandas', pandas.__version__, "
         "'| torch', torch.__version__, '| torchmetrics', torchmetrics.__version__, '| monai', monai.__version__)")
env_chk = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True)
if env_chk.returncode != 0:
    print(env_chk.stderr[-2500:])
    raise SystemExit("!! environment check failed (see the error above) -> send Claude this block")
print("environment OK:", env_chk.stdout.strip())
p = "/kaggle/working/U24_synthesis/lightning_synthesis/model_architectures/monai_ddpm.py"
s = open(p).read()
s2 = re.sub(r"use_flash_attention\s*=\s*True", "use_flash_attention=False", s)   # xformers is not installed
if s2 != s:
    open(p, "w").write(s2)
print("flash attention ->", "patched off" if s2 != s else "already off")
m = "/kaggle/working/U24_synthesis/models.py"          # oracle loader (used later in A6): hard-coded config path
s = open(m).read()
s2 = re.sub(r"open\(\s*['\"]/home/[^'\"]*config\.yaml['\"]", "open('/kaggle/working/U24_synthesis/config.yaml'", s)
if s2 != s:
    open(m, "w").write(s2)
print("Cell 2 done. Kernel restart NOT needed (all GPU work runs in subprocesses).")

In [ ]:
# ===================== CELL 3 — code + clean data + inputs + job runner =====================
import os, sys, glob, json, shutil, subprocess, time
from pathlib import Path
WD = "/kaggle/working/U24_synthesis/lightning_synthesis"
os.chdir(WD)
FILES = {"phaseA_common.py": r'''# phaseA_common.py - shared helpers for the Phase A diagnostics (no training happens in Phase A).
# Model, schedule, data domain and conditioning are taken from the Experiment-2 code (train_arm2.py / eval_arm.py):
#   * model: model_architectures.MonaiDDPM().unet (MONAI-Generative DiffusionModelUNet, flash attention off)
#   * schedule: MonaiDDPM().scheduler.alphas_cumprod (linear beta 1e-4..0.02, T=1000)
#   * images: per-image min-max to [-1,1]; physics statistic on (x+1)/2
#   * conditioning: context = label as float (B,1,1); -1 = null label (classifier-free guidance)
import os, sys, glob, json, math, time
import numpy as np
import torch

WD = "/kaggle/working/U24_synthesis/lightning_synthesis"
if WD not in sys.path:
    sys.path.insert(0, WD)
OUT = os.environ.get("PA_OUT", "/kaggle/working/phaseA")
os.makedirs(OUT, exist_ok=True)


def log(tag, *m):
    print(f"[{tag}]", *m, flush=True)


_INDEX = None


def find(name, roots=("/kaggle/input", "/kaggle/working")):
    """all files with this exact name in the attached inputs (one directory walk per process; image folders skipped)"""
    global _INDEX
    if _INDEX is None:
        _INDEX = {}
        for r in roots:
            for dp, dns, fns in os.walk(r):
                dns[:] = [d for d in dns if d not in ("jpeg", "U24_synthesis", ".git", "phaseA")]
                for fn in fns:
                    _INDEX.setdefault(fn, []).append(os.path.join(dp, fn))
    return sorted(set(_INDEX.get(name, [])))


def main_ckpt():
    """full-data Experiment-2 baseline, epoch 40 (the main Phase A checkpoint)"""
    h = find("base2_w_ep040.pt")
    if not h:
        raise SystemExit("base2_w_ep040.pt not found: Add Input -> Your Work -> the Experiment 2 notebook "
                         "(notebook64a7f79592), then run again")
    return h[0]


def clean_ckpt(seed, epoch=40):
    """clean-data baseline (step 2 of the timeline): base_clean_s<seed>_w_ep<epoch>.pt"""
    h = find(f"base_clean_s{seed}_w_ep{epoch:03d}.pt")
    if not h:
        raise SystemExit(f"base_clean_s{seed}_w_ep{epoch:03d}.pt not found: Add Input -> Your Work -> the clean-baseline "
                         "notebook (its latest output version), then run again")
    return h[0]


def optional_25_ckpt():
    """any 25 %-data baseline at epoch 45 from Experiment 3 or 4 (optional)"""
    for s in (42, 43, 44, 45, 46, 47, 48):
        h = find(f"base3_s{s}_w_ep045.pt")
        if h:
            return h[0]
    return None


def schedule(dev):
    from model_architectures import MonaiDDPM
    m = MonaiDDPM(lr=1e-5, T=1000)
    ac = m.scheduler.alphas_cumprod.detach().double().to(dev)
    unet = m.unet
    del m
    return unet, ac


def load_model(path, dev, dtype=torch.float32):
    """returns (unet in eval mode, alphas_cumprod float64, snapshot meta)"""
    snap = torch.load(path, map_location="cpu", weights_only=False)
    unet, ac = schedule(dev)
    sd = snap["unet"] if "unet" in snap else snap["models"]["net"]
    unet.load_state_dict({k: v.float() for k, v in sd.items()})
    unet = unet.to(dev).to(dtype).eval()
    for p in unet.parameters():
        p.requires_grad_(False)
    meta = {k: snap[k] for k in ("epoch", "step", "lam", "tau") if k in snap}
    meta["nlf"] = snap.get("nlf")
    meta["path"] = path
    del snap
    return unet, ac, meta


def load_cache(split):
    d = np.load(f"/tmp/{split}_cache.npz")
    return d["x"], d["y"]


def load_clean_meta(split):
    """clean cache WITH patient ids and md5 - same order as /tmp/<split>_cache.npz (written by prepare_clean.py)"""
    d = np.load(f"/kaggle/working/clean_data/{split}_cache.npz")
    return d["x"], d["y"], d["patient"].astype(str), d["md5"].astype(str)


def val_mask(patients, frac=0.15, seed=0):
    """patient-level VALIDATION split of the clean TRAIN set (classifiers, gates, guidance choice only).
    The diffusion model itself was trained on all of train. Fixed seed -> identical in every script/session."""
    ups = sorted(set(patients))
    rng = np.random.default_rng(seed)
    val = set(rng.choice(ups, int(round(frac * len(ups))), replace=False).tolist())
    return np.array([p in val for p in patients])


def minmax_pm1(x):
    """x: torch (B,1,H,W) any range -> per-image min-max to [-1,1] (training domain)"""
    mn = x.amin(dim=(1, 2, 3), keepdim=True)
    mx = x.amax(dim=(1, 2, 3), keepdim=True)
    return 2 * (x - mn) / (mx - mn + 1e-8) - 1


def to_model_domain(x_u8, dev):
    return minmax_pm1(torch.from_numpy(x_u8).float().unsqueeze(1).to(dev))


def ctx_of(labels, dev, dtype=torch.float32):
    """labels: list/array of 0, 1 or -1 (null) -> context tensor (B,1,1)"""
    return torch.as_tensor(np.asarray(labels, dtype=np.float32), device=dev).view(-1, 1, 1).to(dtype)


def eps_pred(unet, x, t_int, ctx, amp=False, chunk=8):
    """epsilon prediction. amp=True: exactly as in training (fp32 weights + fp16 autocast); amp=False: the weights'
    own dtype. Large batches are split into chunks (the non-flash attention at 64x64 needs ~0.5 GB per image)."""
    if chunk and x.shape[0] > chunk:
        if ctx.shape[0] == 1:
            ctx = ctx.expand(x.shape[0], *ctx.shape[1:])
        return torch.cat([eps_pred(unet, x[i:i + chunk], t_int, ctx[i:i + chunk], amp, None)
                          for i in range(0, x.shape[0], chunk)])
    tb = torch.full((x.shape[0],), int(t_int), device=x.device, dtype=torch.long)
    if amp:
        with torch.autocast("cuda", dtype=torch.float16):
            out = unet(x, timesteps=tb, context=ctx)
        return out.float()
    return unet(x, timesteps=tb, context=ctx)


def save_json(obj, name):
    with open(os.path.join(OUT, name), "w") as fh:
        json.dump(obj, fh, indent=1, default=float)
''',
         "a6_classifier.py": r'''# a6_classifier.py - A6: real-data benign/malignant classifier on the CLEAN split (+ shared training helpers).
# Used as (i) the "real-only" reference of the later downstream test, (ii) the judge of class conditioning (A7/A9),
# (iii) the "mammography feature" extractor for KID-mammo.
# PRE-STATED RECIPE (no tuning, no epoch selection): torchvision ConvNeXt-Tiny with ImageNet-1k weights,
# 256 px grayscale -> 3 channels, per-image min-max, ImageNet normalisation, AdamW lr 1e-4 wd 0.05, batch 32,
# 15 epochs, cosine schedule with 1 warm-up epoch, horizontal flip + small affine, class-balanced cross-entropy,
# fp16 autocast. FINAL epoch is used. Seeds 0, 1, 2. Trained on clean TRAIN minus the patient-level VALIDATION split.
# PRE-STATED RULE (validation only - the test set is never used for a decision):
#   usable as judge if the mean VALIDATION AUC over the 3 seeds is >= 0.65.  Test AUC is reported, not used.
import os, sys, time, json, math, argparse
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from phaseA_common import log, save_json, load_cache, load_clean_meta, val_mask, OUT

TAG = "A6"
dev = torch.device("cuda")
torch.backends.cudnn.benchmark = True
MEAN = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)


def prep(x01):
    """(B,1,H,W) any range -> per-image min-max -> 3ch ImageNet-normalised"""
    mn = x01.amin(dim=(1, 2, 3), keepdim=True); mx = x01.amax(dim=(1, 2, 3), keepdim=True)
    x = (x01 - mn) / (mx - mn + 1e-8)
    return (x.repeat(1, 3, 1, 1) - MEAN) / STD


def build():
    import torchvision
    m = torchvision.models.convnext_tiny(weights="IMAGENET1K_V1")
    m.classifier[2] = nn.Linear(m.classifier[2].in_features, 2)
    return m


def auc(scores, labels):
    """Mann-Whitney AUC with tie handling (no sklearn dependency)"""
    s, y = np.asarray(scores, float), np.asarray(labels, int)
    order = np.argsort(s); r = np.empty(len(s)); r[order] = np.arange(1, len(s) + 1)
    for v in np.unique(s):
        m = s == v
        if m.sum() > 1:
            r[m] = r[m].mean()
    n1, n0 = (y == 1).sum(), (y == 0).sum()
    return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


@torch.no_grad()
def predict(m, X_u8, bs=64):
    m.eval(); out = []
    for i in range(0, len(X_u8), bs):
        xb = torch.as_tensor(X_u8[i:i + bs]).to(dev).float().div(255).unsqueeze(1)
        with torch.autocast("cuda", dtype=torch.float16):
            out.append(torch.softmax(m(prep(xb)).float(), 1)[:, 1].cpu())
    return torch.cat(out).numpy()


def augment(x, g):
    B = x.shape[0]
    u = lambda: torch.rand(B, generator=g, device=dev) * 2 - 1
    sx, sy, shx, shy = 1 + 0.05 * u(), 1 + 0.05 * u(), 0.05 * u(), 0.05 * u()
    z = torch.zeros(B, device=dev)
    th = torch.stack([torch.stack([sx, shx, z], 1), torch.stack([shy, sy, z], 1)], 1)
    x = F.grid_sample(x, F.affine_grid(th, list(x.shape), align_corners=False), padding_mode="zeros", align_corners=False)
    flip = torch.rand(B, generator=g, device=dev) < 0.5
    return torch.where(flip.view(-1, 1, 1, 1), x.flip(-1), x)


def train_clf(X_u8, y, seed, epochs=15, bs=32, tag=TAG):
    """the fixed recipe above; returns the model after the FINAL epoch"""
    t0 = time.time()
    Xg, Yg = torch.from_numpy(X_u8).to(dev), torch.from_numpy(np.asarray(y)).long().to(dev)
    w = torch.tensor([len(y) / (2 * max(1, (np.asarray(y) == c).sum())) for c in (0, 1)], device=dev, dtype=torch.float32)
    torch.manual_seed(seed)
    m = build().to(dev)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-4, weight_decay=0.05)
    spe = math.ceil(len(y) / bs); total = spe * epochs
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / spe) * 0.5 * (1 + math.cos(math.pi * s / total)))
    scaler = torch.amp.GradScaler("cuda")
    g = torch.Generator(device=dev).manual_seed(seed)
    for ep in range(epochs):
        m.train()
        perm = torch.randperm(len(y), generator=torch.Generator().manual_seed(seed * 100 + ep)).to(dev)
        tl = 0.0
        for i in range(spe):
            idx = perm[i * bs:(i + 1) * bs]
            xb = augment(Xg[idx].float().div(255).unsqueeze(1), g)
            with torch.autocast("cuda", dtype=torch.float16):
                loss = F.cross_entropy(m(prep(xb)).float(), Yg[idx], weight=w)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tl += float(loss)
        if (ep + 1) % 5 == 0 or ep == 0:
            log(tag, f"seed {seed} epoch {ep + 1}/{epochs} train loss {tl / spe:.4f} | {(time.time() - t0) / 60:.1f} min")
    del Xg, Yg
    return m


def boot_auc(p, y, n=1000, seed=0):
    r = np.random.default_rng(seed); vals = []
    i0, i1 = np.where(np.asarray(y) == 0)[0], np.where(np.asarray(y) == 1)[0]
    for _ in range(n):
        b = np.concatenate([r.choice(i0, len(i0)), r.choice(i1, len(i1))])
        vals.append(auc(p[b], np.asarray(y)[b]))
    return [float(np.quantile(vals, 0.025)), float(np.quantile(vals, 0.975))]


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--seeds", default="0,1,2")
    ap.add_argument("--epochs", type=int, default=15)
    a = ap.parse_args()
    x, y, pat, _ = load_clean_meta("train"); xte, yte = load_cache("test")
    vm = val_mask(pat)
    res = {"recipe": "convnext_tiny IMAGENET1K_V1, AdamW 1e-4 wd .05, bs 32, 15 ep cosine (1 warm-up), flip+affine, "
                     "balanced CE, final epoch; trained on train minus validation patients",
           "n_train_fit": int((~vm).sum()), "n_val": int(vm.sum()), "n_val_patients": len(set(pat[vm])),
           "n_test": int(len(yte)), "seeds": {}}
    log(TAG, f"fit on {res['n_train_fit']} images | validation {res['n_val']} images ({res['n_val_patients']} patients)")
    pv_sum = np.zeros(int(vm.sum()))
    for seed in [int(s) for s in a.seeds.split(",")]:
        t0 = time.time()
        m = train_clf(x[~vm], y[~vm], seed, a.epochs)
        pv, pt = predict(m, x[vm]), predict(m, xte)
        pv_sum += pv
        res["seeds"][str(seed)] = {"val_auc": auc(pv, y[vm]), "test_auc_reported_only": auc(pt, yte),
                                   "val_balanced_acc": float(np.mean([((pv >= 0.5) == (y[vm] == 1))[y[vm] == c].mean() for c in (0, 1)])),
                                   "minutes": (time.time() - t0) / 60}
        torch.save({k: v.half() for k, v in m.state_dict().items()}, os.path.join(OUT, f"a6_clf_s{seed}.pt"))
        log(TAG, f"seed {seed}: VALIDATION AUC {res['seeds'][str(seed)]['val_auc']:.3f} | test AUC (reported only) "
                 f"{res['seeds'][str(seed)]['test_auc_reported_only']:.3f} | {(time.time() - t0) / 60:.1f} min")
        del m; torch.cuda.empty_cache()
    va = [v["val_auc"] for v in res["seeds"].values()]
    res["mean_val_auc"] = float(np.mean(va)); res["sd_val_auc"] = float(np.std(va, ddof=1)) if len(va) > 1 else 0.0
    pe = pv_sum / len(va)
    res["ensemble_val_auc"] = auc(pe, y[vm]); res["ensemble_val_auc_ci95"] = boot_auc(pe, y[vm])
    res["mean_test_auc_reported_only"] = float(np.mean([v["test_auc_reported_only"] for v in res["seeds"].values()]))
    res["usable_as_judge"] = bool(res["mean_val_auc"] >= 0.65)
    save_json(res, "A6_classifier.json")
    log(TAG, f"mean VALIDATION AUC {res['mean_val_auc']:.3f} +- {res['sd_val_auc']:.3f} (ensemble {res['ensemble_val_auc']:.3f}) "
             f"-> usable as judge: {res['usable_as_judge']}")
''',
         "pinn_physics.py": r'''# pinn_physics.py - physics-informed soft constraint (PINN-style residual) for mammogram DDPM.
# Physical model: signal-dependent noise-level function (NLF) of the X-ray film/digitiser chain
#   sigma^2(mu) = b + a*mu + c*mu^2
#   b: Gaussian read/digitiser noise, a*mu: Poisson (quantum) noise, c*mu^2: multiplicative film-grain/gain term
# The coefficients are CALIBRATED on real training images, then the residual between the NLF of a
# generated image and the physical NLF is added to the diffusion loss (lambda * residual).
import torch
import torch.nn.functional as F


class PhysicsNLF:
    def __init__(self, device, k=7, K=16, lo=0.08, hi=0.95, mask_lo=0.04):
        ax = torch.arange(7, device=device, dtype=torch.float32) - 3
        g = torch.exp(-ax ** 2 / 2.0)
        self.g = g / g.sum()
        self.k, self.K, self.lo, self.hi, self.mask_lo = k, K, lo, hi, mask_lo
        self.c = torch.linspace(lo, hi, K, device=device)
        self.bw = float(hi - lo) / (K - 1) * 0.6
        self.p = None                     # (b, a, c) of the calibrated NLF

    def blur(self, I):
        g = self.g
        I = F.conv2d(F.pad(I, (3, 3, 0, 0), mode="reflect"), g.view(1, 1, 1, 7))
        return F.conv2d(F.pad(I, (0, 0, 3, 3), mode="reflect"), g.view(1, 1, 7, 1))

    def stats(self, I):
        # I: (B,1,H,W) intensities in [0,1]. Returns per-image binned high-frequency variance vs local mean.
        h = I - self.blur(I)
        mu = F.avg_pool2d(I, self.k, 1, self.k // 2, count_include_pad=False).flatten(1)
        v = F.avg_pool2d(h * h, self.k, 1, self.k // 2, count_include_pad=False).flatten(1)
        W = torch.exp(-(mu.unsqueeze(-1) - self.c) ** 2 / (2 * self.bw ** 2))
        W = W * (mu > self.mask_lo).unsqueeze(-1).float()        # ignore black background (air)
        mass = W.sum(1)                                           # (B,K)
        vbin = torch.einsum("bpk,bp->bk", W, v) / (mass + 1e-6)   # (B,K)
        return vbin, mass

    def model_var(self):
        b, a, c = self.p
        return (b + a * self.c + c * self.c ** 2).clamp_min(1e-7)

    def residual(self, I):
        # per-image physics residual: mass-weighted squared log-ratio of measured vs physical NLF
        vbin, mass = self.stats(I)
        r = (torch.log(vbin + 1e-5) - torch.log(self.model_var() + 1e-5)) ** 2
        w = mass / (mass.sum(1, keepdim=True) + 1e-6)
        return (w * r).sum(1)

    @torch.no_grad()
    def fit(self, X_u8, bs=32):
        dev = self.c.device
        tv = torch.zeros(self.K, device=dev, dtype=torch.float64)
        tm = torch.zeros(self.K, device=dev, dtype=torch.float64)
        for i in range(0, X_u8.shape[0], bs):
            I = X_u8[i:i + bs].to(dev).float().div(255).unsqueeze(1)
            vb, m = self.stats(I)
            tv += (vb * m).sum(0).double()
            tm += m.sum(0).double()
        V = tv / tm.clamp_min(1e-6)
        wt = tm / tm.sum()
        c = self.c.double()

        def wls(cols):
            A = torch.stack(cols, 1)
            th = torch.linalg.solve(A.T @ (wt[:, None] * A), A.T @ (wt * V))
            pred = (A @ th).clamp_min(1e-9)
            ss_res = (wt * (V - pred) ** 2).sum()
            ss_tot = (wt * (V - (wt * V).sum()) ** 2).sum()
            lr = torch.log(V) - torch.log(pred)
            lres = (wt * lr ** 2).sum()
            ltot = (wt * (torch.log(V) - (wt * torch.log(V)).sum()) ** 2).sum()
            return th, float(1 - ss_res / ss_tot), float(1 - lres / ltot)

        th_lin, r2_lin, lr2_lin = wls([torch.ones_like(c), c])
        th_q, r2_q, lr2_q = wls([torch.ones_like(c), c, c ** 2])
        self.p = (float(th_q[0]), float(th_q[1]), float(th_q[2]))
        return {"model": "sigma^2 = b + a*mu + c*mu^2 (Poisson-Gaussian + multiplicative)",
                "b": self.p[0], "a": self.p[1], "c": self.p[2],
                "R2_var": r2_q, "R2_logvar": lr2_q,
                "linear_poisson_gaussian_only": {"b": float(th_lin[0]), "a": float(th_lin[1]),
                                                  "R2_var": r2_lin, "R2_logvar": lr2_lin},
                "bin_centers": [float(x) for x in c], "measured_var": [float(x) for x in V],
                "bin_weight": [float(x) for x in wt]}

    @torch.no_grad()
    def mean_residual(self, X_u8, bs=32):
        dev = self.c.device
        tot, n = 0.0, 0
        for i in range(0, X_u8.shape[0], bs):
            I = X_u8[i:i + bs].to(dev).float().div(255).unsqueeze(1)
            tot += float(self.residual(I).sum())
            n += I.shape[0]
        return tot / max(n, 1)

    def state(self):
        return {"p": list(self.p), "k": self.k, "K": self.K, "lo": self.lo, "hi": self.hi, "mask_lo": self.mask_lo}

    def load(self, s):
        self.p = tuple(s["p"])
''',
         "s7_noise.py": r'''# s7_noise.py - S7 "physics-based acquisition layer" (see S7_preregistration.txt + Addendum 1; nothing here is tuned).
#   1. fit the Poisson-Gaussian noise-level function sigma^2(I) = a*I + b on the train-fit REAL images
#      (pinn_physics.PhysicsNLF, linear fit, per-image min-max normalised images - exactly as in S5)
#   2. for every generated set: I = per-image min-max to [0,1]; I_out = clip(I + sqrt(max(a*I + b, 0)) * z, 0, 1),
#      z ~ N(0,1) from one fixed seed; rounded back to uint8 -> gen_<name><suffix>.npz
import os, sys, argparse
import numpy as np, torch
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from phaseA_common import log, save_json, load_clean_meta, val_mask, OUT
from pinn_physics import PhysicsNLF

ap = argparse.ArgumentParser()
ap.add_argument("--sets", required=True)            # comma list of gen_<name>.npz names
ap.add_argument("--seed", type=int, required=True)  # noise seed (validation 7007, test 9707)
ap.add_argument("--suffix", default="_nl")
a = ap.parse_args()
TAG = "S7-noise"
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def mm01(X_u8):
    X = X_u8.astype(np.float64); mn = X.min(axis=(1, 2), keepdims=True); mx = X.max(axis=(1, 2), keepdims=True)
    return (X - mn) / (mx - mn + 1e-8)

xtr, ytr, pat, _ = load_clean_meta("train"); XF = xtr[~val_mask(pat)]
nlf = PhysicsNLF(dev)
rep = nlf.fit(torch.from_numpy(np.round(mm01(XF) * 255).clip(0, 255).astype(np.uint8)))
lin = rep["linear_poisson_gaussian_only"]; A, B = lin["a"], lin["b"]
log(TAG, f"noise-level function fitted on {len(XF)} train-fit images: sigma^2 = {A:.4e} * I + {B:.4e} "
         f"(linear Poisson-Gaussian, R2 {lin['R2_var']:.3f}) | quadratic fit for reference: b {rep['b']:.3e} a {rep['a']:.3e} c {rep['c']:.3e}")
if A < 0:
    log(TAG, "NOTE: fitted Poisson coefficient a < 0 (variance falls with brightness) - used as fitted (pre-stated), sigma^2 clamped at 0")
save_json({"a": A, "b": B, "fit": rep, "seed": a.seed}, f"S7_nlf_seed{a.seed}.json")

rng = np.random.default_rng(a.seed)
for nm in a.sets.split(","):
    d = np.load(os.path.join(OUT, f"gen_{nm}.npz"))
    I = mm01(d["x"])
    sig = np.sqrt(np.maximum(A * I + B, 0.0))
    Io = np.clip(I + sig * rng.standard_normal(I.shape), 0.0, 1.0)
    Xo = np.round(Io * 255).astype(np.uint8)
    np.savez_compressed(os.path.join(OUT, f"gen_{nm}{a.suffix}.npz"), x=Xo, y=d["y"])
    log(TAG, f"{nm} -> gen_{nm}{a.suffix}.npz | {len(Xo)} images | mean added noise std {sig.mean():.4f} "
             f"(= {255 * sig.mean():.1f} grey levels)")
''',
         "s7_stats.py": r'''# s7_stats.py - copy of s6_stats.py with a selectable CI level (alpha). S6 statistics (pure numpy). Pre-stated rule (fixed 2026-10-21 ~17:10 Asia/Dhaka, before any S6 result):
#   For every arm vs the baseline arm (same noise bank -> generated image i is paired across arms):
#     KID-mammo  = unbiased MMD^2 (cubic kernel) between the generated set and the reference set in the A6 mammography-
#                  classifier feature space; KID-dino = the same in DINOv2 ViT-B/14 feature space (independent "eye")
#     precision / recall (Kynkaenniemi et al. 2019, k = 3) in DINOv2 space: realism / coverage (diversity)
#   PASS if ALL: KID-mammo <= 0.90 x baseline with the paired-bootstrap 95 % CI of the difference < 0;
#               KID-dino <= 1.00 x baseline (no loss in the independent feature space);
#               recall >= baseline recall - 0.05 (no loss of diversity).
#   Validation stage: reference = train-fit images; WINNER = passing arm with the lowest KID-mammo (none -> no winner).
#   Test stage (run ONCE, frozen winner vs baseline, fresh noise seeds): reference = the untouched TEST images; the same
#   rule -> CONFIRMED / NOT CONFIRMED.
import numpy as np
from scipy.spatial import cKDTree
from scipy.spatial.distance import cdist

kern = lambda A_, B_: (A_ @ B_.T / A_.shape[1] + 1.0) ** 3

def _kRR(R):
    K = kern(R, R); return (K.sum() - np.trace(K)) / (len(R) * (len(R) - 1))

def _mmd(KGG, h, kRR, c):
    num = c @ KGG @ c - (c * c * np.diag(KGG)).sum(); den = c.sum() ** 2 - (c * c).sum()
    return float(num / den - 2 * (c * h).sum() / c.sum() + kRR)

def prec_recall(real, fake, k=3):
    """improved precision / recall (Kynkaenniemi et al. 2019): k-NN balls around real (fake) points"""
    rr = cKDTree(real).query(real, k=k + 1)[0][:, -1]; rf = cKDTree(fake).query(fake, k=k + 1)[0][:, -1]
    D = cdist(fake, real)
    return float((D <= rr[None, :]).any(1).mean()), float((D.T <= rf[None, :]).any(1).mean())

def evaluate(F, base, n_boot=1000, seed=0, alpha=0.05):
    """F: {"ref_mammo", "ref_dino", "<arm>_mammo", "<arm>_dino", "<arm>_istats" (optional)}; returns (lines, results)"""
    arms = sorted({k[:-6] for k in F if k.endswith("_mammo") and not k.startswith("ref")})
    assert base in arms, f"baseline {base} missing"
    n = min(len(F[f"{a}_mammo"]) for a in arms)
    C = np.random.default_rng(seed).multinomial(n, np.ones(n) / n, size=n_boot).astype(float); ones = np.ones(n)
    res = {}
    for space in ("mammo", "dino"):
        R = F[f"ref_{space}"]; kRR = _kRR(R)
        for a in arms:
            G = F[f"{a}_{space}"][:n]; KGG, h = kern(G, G), kern(G, R).mean(1)
            res.setdefault(a, {})[f"kid_{space}"] = _mmd(KGG, h, kRR, ones)
            res[a][f"boot_{space}"] = np.array([_mmd(KGG, h, kRR, c) for c in C])
    for a in arms:
        res[a]["precision"], res[a]["recall"] = prec_recall(F["ref_dino"], F[f"{a}_dino"][:n])
        if f"{a}_istats" in F:
            st = F[f"{a}_istats"][:n]; res[a]["saturation"], res[a]["hf_energy"] = float(st[:, 1].mean()), float(st[:, 2].mean())
    b = res[base]; L = []; out = {}
    L.append(f"  baseline {base}: KID-mammo {b['kid_mammo']:.4f} | KID-dino {b['kid_dino']:.4f} | precision {b['precision']:.3f} "
             f"recall {b['recall']:.3f}" + (f" | saturation {b['saturation']:.4f} HF {b['hf_energy']:.4f}" if "saturation" in b else ""))
    for a in arms:
        r = res[a]
        if a == base:
            out[a] = {k: v for k, v in r.items() if not k.startswith("boot")}; continue
        dm = r["boot_mammo"] - b["boot_mammo"]; dd = r["boot_dino"] - b["boot_dino"]
        lo, hi = alpha / 2, 1 - alpha / 2
        ci_m = [float(np.quantile(dm, lo)), float(np.quantile(dm, hi))]; ci_d = [float(np.quantile(dd, lo)), float(np.quantile(dd, hi))]
        ok = {"KIDm<=0.90x": bool(r["kid_mammo"] <= 0.90 * b["kid_mammo"]), "KIDm_CI<0": bool(ci_m[1] < 0),
              "KIDd<=1.00x": bool(r["kid_dino"] <= b["kid_dino"]), "recall_ok": bool(r["recall"] >= b["recall"] - 0.05)}
        out[a] = {"kid_mammo": r["kid_mammo"], "kid_mammo_ratio": r["kid_mammo"] / b["kid_mammo"], "kid_mammo_diff_ci95": ci_m,
                  "kid_dino": r["kid_dino"], "kid_dino_ratio": r["kid_dino"] / b["kid_dino"], "kid_dino_diff_ci95": ci_d,
                  "precision": r["precision"], "recall": r["recall"], "saturation": r.get("saturation"), "hf_energy": r.get("hf_energy"),
                  "checks": ok, "passed": all(ok.values())}
        L.append(f"  {a:16s}: KID-mammo x{out[a]['kid_mammo_ratio']:.3f} (diff CI {ci_m[0]:+.4f}..{ci_m[1]:+.4f}) | KID-dino "
                 f"x{out[a]['kid_dino_ratio']:.3f} (CI {ci_d[0]:+.4f}..{ci_d[1]:+.4f}) | precision {r['precision']:.3f} recall {r['recall']:.3f}"
                 + (f" | saturation {r['saturation']:.4f} HF {r['hf_energy']:.4f}" if "saturation" in r else "")
                 + f" -> {'PASS' if out[a]['passed'] else 'fail'} {ok}")
    passed = sorted([(v["kid_mammo_ratio"], a) for a, v in out.items() if v.get("passed")])
    return L, out, (passed[0][1] if passed else None)

def _auc(p, y):
    from scipy.stats import rankdata
    r = rankdata(p); n1 = (y == 1).sum(); n0 = (y == 0).sum()
    return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))

def downstream_summary(D, ref="real_only", n_boot=2000, seed=0):
    """D: {arm: {"y": y, "p0": ..., "p1": ..., ...}} -> lines (secondary, descriptive)"""
    L = []; rng = np.random.default_rng(seed)
    y = D[ref]["y"]; i0, i1 = np.where(y == 0)[0], np.where(y == 1)[0]
    B = [np.concatenate([rng.choice(i0, len(i0)), rng.choice(i1, len(i1))]) for _ in range(n_boot)]
    ens = {a: np.mean([v for k, v in d.items() if k.startswith("p")], 0) for a, d in D.items()}
    for a, d in D.items():
        aucs = [_auc(v, y) for k, v in d.items() if k.startswith("p")]
        line = f"  {a:16s}: TEST AUC {np.mean(aucs):.4f} +- {np.std(aucs, ddof=1) if len(aucs) > 1 else 0:.4f} (seeds) | ensemble {_auc(ens[a], y):.4f}"
        if a != ref:
            diff = [_auc(ens[a][b], y[b]) - _auc(ens[ref][b], y[b]) for b in B]
            line += f" | vs {ref}: {_auc(ens[a], y) - _auc(ens[ref], y):+.4f} (CI {np.quantile(diff, .025):+.4f}..{np.quantile(diff, .975):+.4f})"
        L.append(line)
    return L
''',
         "s7_eval.py": r'''# s7_eval.py - s6_eval.py with --tag (output file prefix) and --alpha (CI level). S6 evaluation on the GPU (features) + s6_stats.evaluate (pre-stated rule, see s6_stats.py header).
#   --stage val : reference = train-fit images (validation split never used for training decisions); writes the WINNER
#   --stage test: reference = the untouched TEST images (run ONCE on the frozen winner vs baseline, fresh noise seeds)
import os, sys, json, argparse
import numpy as np, torch, torch.nn.functional as F
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from phaseA_common import log, save_json, load_clean_meta, load_cache, val_mask, find, OUT
from a6_classifier import build, prep
from s7_stats import evaluate

ap = argparse.ArgumentParser()
ap.add_argument("--stage", choices=["val", "test"], required=True)
ap.add_argument("--sets", required=True)            # comma list, the FIRST is the baseline
ap.add_argument("--n_boot", type=int, default=1000)
ap.add_argument("--tag", default="S7")
ap.add_argument("--alpha", type=float, default=0.05)
a = ap.parse_args()
TAG = f"{a.tag}-eval-{a.stage}"
dev = torch.device("cuda")
names = a.sets.split(","); base = names[0]
if a.stage == "val":
    xtr, ytr, pat, _ = load_clean_meta("train"); REF = xtr[~val_mask(pat)]
else:
    REF, _ = load_cache("test")
log(TAG, f"reference: {'train-fit' if a.stage == 'val' else 'TEST'} images {REF.shape} | sets {names}")

mam = build(); mam.load_state_dict({k: v.float() for k, v in torch.load(find("a6_clf_s0.pt")[0], map_location="cpu").items()})
mam = mam.to(dev).eval()
def load_dino():
    try:
        return torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14", trust_repo=True)
    except Exception as e:
        log(TAG, f"torch.hub DINOv2 failed ({repr(e)[:150]}) -> timm")
        import timm
        return timm.create_model("vit_base_patch14_dinov2.lvd142m", pretrained=True, num_classes=0, img_size=224)
dino = load_dino().to(dev).eval()
MEAN = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1); STD = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)
@torch.no_grad()
def feats(X_u8, bs=64):
    fm, fd = [], []
    for i in range(0, len(X_u8), bs):
        xb = torch.from_numpy(X_u8[i:i + bs]).to(dev).float().div(255).unsqueeze(1)
        with torch.autocast("cuda", dtype=torch.float16):
            fm.append(mam.classifier[1](mam.classifier[0](mam.avgpool(mam.features(prep(xb))))).float().cpu())
        z = F.interpolate(xb, size=224, mode="bilinear", align_corners=False).repeat(1, 3, 1, 1)
        fd.append(dino((z - MEAN) / STD).float().cpu())
    return torch.cat(fm).numpy().astype(np.float64), torch.cat(fd).numpy().astype(np.float64)
def istats_img(X_u8):
    I = torch.from_numpy(X_u8).float().div(255).unsqueeze(1)
    mn = I.amin(dim=(1, 2, 3), keepdim=True); mx = I.amax(dim=(1, 2, 3), keepdim=True); I = (I - mn) / (mx - mn + 1e-8)
    mu = F.avg_pool2d(I, 7, 1, 3, count_include_pad=False); m = (mu > 0.04).float(); n = m.sum((1, 2, 3)).clamp_min(1)
    hp = I - F.avg_pool2d(I, 7, 1, 3, count_include_pad=False)
    return np.stack([((I * m).sum((1, 2, 3)) / n).numpy(), ((((I > 0.95).float() * m).sum((1, 2, 3))) / n).numpy(),
                     ((hp * hp * m).sum((1, 2, 3)) / n).numpy(), m.mean((1, 2, 3)).numpy()], 1)

Fd = {}
Fd["ref_mammo"], Fd["ref_dino"] = feats(REF); Fd["ref_istats"] = istats_img(REF)
for nm in names:
    d = np.load(os.path.join(OUT, f"gen_{nm}.npz"))
    Fd[f"{nm}_mammo"], Fd[f"{nm}_dino"] = feats(d["x"]); Fd[f"{nm}_istats"] = istats_img(d["x"])
np.savez(os.path.join(OUT, f"{a.tag}_{a.stage}_features.npz"), **Fd)
L, out, win = evaluate(Fd, base, a.n_boot, alpha=a.alpha)
ri = Fd["ref_istats"]
head = [f"[{a.tag} {a.stage.upper()}] CI level {100 * (1 - a.alpha):.1f} % | reference {'train-fit' if a.stage == 'val' else 'TEST'} images: saturation {ri[:, 1].mean():.4f} HF {ri[:, 2].mean():.4f}"]
if a.stage == "val":
    tail = [f"  -> WINNER: {win}" if win else "  -> no arm passed: no winner (test stage compares the baseline only descriptively)"]
    save_json({"winner": win, "results": out}, f"{a.tag}_decision.json")
else:
    w = [n for n in names if n != base]
    ok = bool(w and out[w[0]].get("passed"))
    tail = [f"  -> TEST CONFIRMATION of {w[0] if w else '-'}: {'CONFIRMED' if ok else 'NOT CONFIRMED'}"]
    save_json({"confirmed": ok, "results": out}, f"{a.tag}_test.json")
txt = "\n".join(head + L + tail)
open(os.path.join(OUT, f"{a.tag}_{a.stage}_report.txt"), "w").write(txt)
print(txt, flush=True)
'''}
for n, s in FILES.items():
    Path(n).write_text(s)
Path("/tmp/prepare_clean.py").write_text(r'''# prepare_clean.py - CLEAN patient-level CBIS-DDSM cache (rule B, decided 19 Oct 2026 after the data audit).
#   unit            = one unique image (pixel-identical after the 256x256 resize = the same image), not one CSV row
#   resolution      = PatientName in dicom_info (as before); if missing: series UID of the CSV path -> dicom_info, or
#                     the jpeg folder named by that UID holding exactly one image (recovers the 282 skipped test rows)
#   image check     = native min side >= 1000 px and not a binary mask (a full mammogram), else rejected + reported
#   label conflicts = an image carrying both BENIGN* and MALIGNANT rows -> excluded
#   split (rule B)  = patients that occur in both official train and test CSVs go to TEST (all their images);
#                     an image that occurs in both splits therefore ends up in test; train = everything else
#   no class subsampling (classes are already near-balanced)
# Writes /tmp/train_cache.npz and /tmp/test_cache.npz (x uint8 (N,256,256), y int64) exactly like the old prepare.py,
# plus /kaggle/working/clean_data/ (compressed caches with md5 + patient, manifests, summary) so that EVERY later
# notebook can attach this output and use byte-identical data.
import os, glob, json, shutil, subprocess, hashlib, collections, time
import numpy as np, pandas as pd
from PIL import Image

OUT = "/kaggle/working/clean_data"
os.makedirs(OUT, exist_ok=True)
EXPECTED = {"train": 2336, "test": 663}          # data audit, rule B (before the full-mammogram check)

def to_tmp(src_dir):
    for s in ("train", "test"):
        d = np.load(os.path.join(src_dir, f"{s}_cache.npz"))
        np.savez(f"/tmp/{s}_cache.npz", x=d["x"], y=d["y"])
        print(f"{s}: {d['x'].shape} benign={int((d['y'] == 0).sum())} malignant={int((d['y'] == 1).sum())}")

# ---------- 0. already built in this session, or attached from an earlier session -> reuse (identical bytes) ----------
if os.path.exists(os.path.join(OUT, "clean_summary.json")):
    to_tmp(OUT); print("clean caches already built in this session"); raise SystemExit(0)
prev = sorted(glob.glob("/kaggle/input/**/clean_data/clean_summary.json", recursive=True))
if prev:
    src = os.path.dirname(prev[0])
    for f in os.listdir(src):
        shutil.copy(os.path.join(src, f), os.path.join(OUT, f))
    to_tmp(OUT)
    print("clean caches REUSED from", src); raise SystemExit(0)

t0 = time.time()
hit = subprocess.run("find /kaggle/input -maxdepth 6 -name dicom_info.csv", shell=True,
                     capture_output=True, text=True).stdout.split()
if not hit:
    raise SystemExit("CBIS-DDSM not attached: Add Input -> CBIS-DDSM Breast Cancer Image Dataset")
RAW = os.path.dirname(os.path.dirname(hit[0]))
JPEG = os.path.join(RAW, "jpeg")
info = pd.read_csv(os.path.join(RAW, "csv/dicom_info.csv"))
jp = lambda rel: os.path.join(JPEG, str(rel).replace("CBIS-DDSM/jpeg/", ""))
full = info[info["SeriesDescription"] == "full mammogram images"]
by_name = full.set_index("PatientName")["image_path"].to_dict()
by_uid = full.set_index("SeriesInstanceUID")["image_path"].to_dict()
jpeg_dirs = set(os.listdir(JPEG))

# ---------- 1. every CSV row -> image file ----------
rows, how_n = [], collections.Counter()
for split, csvs in [("train", ["csv/calc_case_description_train_set.csv", "csv/mass_case_description_train_set.csv"]),
                    ("test", ["csv/calc_case_description_test_set.csv", "csv/mass_case_description_test_set.csv"])]:
    for c in csvs:
        for _, r in pd.read_csv(os.path.join(RAW, c)).iterrows():
            parts = str(r["image file path"]).strip().split("/")
            rel, how = by_name.get(parts[0]), "patientname"
            if rel is not None and not os.path.exists(jp(rel)):
                rel = None
            if rel is None:
                for comp in parts[1:3]:
                    if comp in by_uid and os.path.exists(jp(by_uid[comp])):
                        rel, how = by_uid[comp], "series_uid_dicom_info"
                        break
            if rel is None:
                for comp in parts[1:3]:
                    if comp in jpeg_dirs:
                        fs = [f for f in os.listdir(os.path.join(JPEG, comp)) if f.lower().endswith(".jpg")]
                        if len(fs) == 1:
                            rel, how = os.path.join(comp, fs[0]), "uid_folder_single_jpeg"
                            break
            if rel is None:
                how = "unresolved"
            how_n[f"{split}:{how}"] += 1
            rows.append({"split": split, "csv": os.path.basename(c), "patient": str(r["patient_id"]),
                         "label": 1 if r["pathology"] == "MALIGNANT" else 0, "rel": rel, "how": how})
R = pd.DataFrame(rows)
print("row resolution:", dict(how_n), flush=True)
Rk = R[R.rel.notna()].copy()

# ---------- 2. load every unique file once: full-mammogram check + 256x256 array + md5 ----------
arr, md5, native, reject = {}, {}, {}, {}
files = sorted(Rk.rel.unique())
for i, rel in enumerate(files):
    with Image.open(jp(rel)) as im:
        W, H = im.size
        g = im.convert("L")
        native[rel] = (W, H)
        if min(W, H) < 1000:
            reject[rel] = f"small ({W}x{H}) - not a full mammogram"
            continue
        small = np.asarray(g.resize((512, 512)))
        if np.mean((small > 30) & (small < 225)) < 0.02:
            reject[rel] = "binary image - looks like an ROI mask"
            continue
        a = np.array(g.resize((256, 256)), dtype=np.uint8)            # identical resize to the old prepare.py
    arr[rel] = a
    md5[rel] = hashlib.md5(a.tobytes()).hexdigest()
    if (i + 1) % 500 == 0:
        print(f"  {i + 1}/{len(files)} files ({(time.time() - t0) / 60:.1f} min)", flush=True)
rej_rows = Rk[Rk.rel.isin(reject)]
Rk = Rk[~Rk.rel.isin(reject)].copy()
Rk["md5"] = Rk.rel.map(md5)

# ---------- 3. one record per image, conflicts, rule-B split ----------
ptr, pte = set(R[(R.split == "train") & R.rel.notna()].patient), set(R[(R.split == "test") & R.rel.notna()].patient)
both = ptr & pte                                    # same definition as the data audit
img = Rk.groupby("md5").agg(rel=("rel", "first"), files=("rel", lambda s: sorted(set(s))),
                            splits=("split", lambda s: sorted(set(s))), patients=("patient", lambda s: sorted(set(s))),
                            labels=("label", lambda s: sorted(set(s))), csvs=("csv", lambda s: sorted(set(s)))).reset_index()
conflict = img.labels.map(len) > 1
clean = img[~conflict].copy()
clean["label"] = clean.labels.map(lambda s: s[0])
touch = clean.patients.map(lambda ps: any(p in both for p in ps))
in_tr = clean.splits.map(lambda s: "train" in s)
in_te = clean.splits.map(lambda s: "test" in s)
clean["final_split"] = np.where(in_te | touch, "test", "train")
clean = clean.sort_values("md5").reset_index(drop=True)
assert not (set(p for ps in clean[clean.final_split == "train"].patients for p in ps) &
            set(p for ps in clean[clean.final_split == "test"].patients for p in ps)), "patient overlap remains!"

summary = {"rule": "B: patients in both official splits -> test; label-conflict images excluded; one record per "
                   "pixel-identical image; full-mammogram check", "decided": "2026-10-19",
           "row_resolution": dict(how_n), "rejected_files": len(reject), "rejected_examples": dict(list(reject.items())[:10]),
           "rejected_rows": int(len(rej_rows)), "unique_images": int(len(img)), "label_conflict_images_excluded": int(conflict.sum()),
           "patients_in_both_official_splits": len(both), "build_minutes": None}
for s in ("train", "test"):
    d = clean[clean.final_split == s]
    x = np.stack([arr[r] for r in d.rel]); y = d.label.to_numpy().astype(np.int64)
    np.savez(f"/tmp/{s}_cache.npz", x=x, y=y)
    np.savez_compressed(os.path.join(OUT, f"{s}_cache.npz"), x=x, y=y, md5=d.md5.to_numpy().astype(str),
                        patient=d.patients.map(lambda p: "|".join(p)).to_numpy().astype(str))
    m = d[["md5", "label", "patients", "files", "csvs", "splits"]].copy()
    m["native_WxH"] = d.rel.map(lambda r: "x".join(map(str, native[r])))
    m.to_csv(os.path.join(OUT, f"manifest_{s}.csv"), index=False)
    pats = set(p for ps in d.patients for p in ps)
    summary[s] = {"images": int(len(d)), "benign": int((y == 0).sum()), "malignant": int((y == 1).sum()),
                  "patients": len(pats), "expected_from_audit": EXPECTED[s], "difference": int(len(d)) - EXPECTED[s]}
    print(f"{s}: {x.shape} benign={int((y == 0).sum())} malignant={int((y == 1).sum())} patients={len(pats)} "
          f"(audit expected {EXPECTED[s]}, difference {int(len(d)) - EXPECTED[s]})", flush=True)
summary["build_minutes"] = (time.time() - t0) / 60
json.dump(summary, open(os.path.join(OUT, "clean_summary.json"), "w"), indent=1, default=str)
print(f"rejected files (full-mammogram check): {len(reject)}", dict(list(reject.items())[:5]))
print("clean caches written to /tmp and", OUT, f"({summary['build_minutes']:.1f} min)")
''')
subprocess.run([sys.executable, "/tmp/prepare_clean.py"], check=True)
def find_in(name):
    h = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    print(f"{name}: {h[0] if h else '-- not found'}"); return h[0] if h else None
def need(name, hint):
    p = find_in(name)
    if p is None:
        raise SystemExit(f"!! {name} not found -> attach the {hint} notebook output (Add Input)")
    return p
OUT = os.environ["PA_OUT"]
need("a6_clf_s0.pt", "S2")
for nm in ("v_base", "t_base"):
    shutil.copy(need(f"gen_{nm}.npz", "S6 (final)"), f"{OUT}/gen_{nm}.npz")
LOGS = f"{OUT}/logs"; os.makedirs(LOGS, exist_ok=True)
RC = {}
def run_one(name, argv, est_h, out_dir=None):
    if used_h() + est_h > BUDGET_H - 0.2:
        print(f"--- {name} SKIPPED (budget)", flush=True); RC[name] = "skipped"; return
    env = os.environ.copy(); env["PYTHONUNBUFFERED"] = "1"; env["CUDA_VISIBLE_DEVICES"] = "0"
    if out_dir:
        env["PA_OUT"] = out_dir
    print(f">>> {name} started | session {used_h():.2f} h", flush=True)
    with open(f"{LOGS}/{name}.log", "w") as fh:
        p = subprocess.Popen([sys.executable, "-u"] + argv, cwd=WD, env=env, text=True, bufsize=1, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        for line in p.stdout:
            fh.write(line); fh.flush(); print(line if line.startswith("[") else f"[{name}] {line}", end="", flush=True)
        RC[name] = p.wait()
    print(f"<<< {name} return code {RC[name]} | session {used_h():.2f} h" + ("" if RC[name] == 0 else f"  !! see {LOGS}/{name}.log"), flush=True)
print("Cell 3 done.")

In [ ]:
# ===================== CELL 4 — SMOKE TEST (~5 min) =====================
SM = "/kaggle/working/smoke"
if RUN_SMOKE and not os.path.exists(f"{OUT}/S7_decision.json"):
    os.makedirs(SM, exist_ok=True)
    subprocess.run([sys.executable, "-c", "import numpy as np; d = np.load('" + OUT + "/gen_v_base.npz'); "
                    "np.savez('" + SM + "/gen_sm.npz', x=d['x'][::25], y=d['y'][::25])"], check=True)
    run_one("smoke_noise", ["s7_noise.py", "--sets", "sm", "--seed", "1"], 0.05, SM)
    run_one("smoke_eval", ["s7_eval.py", "--stage", "val", "--sets", "sm,sm_nl", "--n_boot", "20"], 0.1, SM)
    if not (RC.get("smoke_noise") == 0 and RC.get("smoke_eval") == 0 and os.path.exists(f"{SM}/S7_decision.json")):
        raise SystemExit("SMOKE TEST FAILED -> nothing long was started. Send Claude this block + S7/logs/smoke_*.log")
    shutil.rmtree(SM, ignore_errors=True); RC.clear(); print(f"SMOKE TEST PASSED | session {used_h():.2f} h")

In [ ]:
# ===================== CELL 5 — VALIDATION (reference = train-fit images) =====================
run_one("noise_val", ["s7_noise.py", "--sets", "v_base", "--seed", str(VAL_SEED)], 0.1)
run_one("eval_val", ["s7_eval.py", "--stage", "val", "--sets", "v_base,v_base_nl"], 0.2)
DEC = json.load(open(f"{OUT}/S7_decision.json")) if os.path.exists(f"{OUT}/S7_decision.json") else {"winner": None}
print("S7 VALIDATION:", "PASS -> one test confirmation" if DEC["winner"] == "v_base_nl" else "FAIL -> no test stage (pre-stated). S7 is negative.")

In [ ]:
# ===================== CELL 6 — TEST (only if validation passed) + report + zip =====================
if DEC["winner"] == "v_base_nl":
    run_one("noise_test", ["s7_noise.py", "--sets", "t_base", "--seed", str(TEST_SEED)], 0.1)
    run_one("eval_test", ["s7_eval.py", "--stage", "test", "--sets", "t_base,t_base_nl", "--alpha", str(TEST_ALPHA)], 0.3)
rep = "\n\n".join(open(f"{OUT}/{f}").read() for f in ("S7_val_report.txt", "S7_test_report.txt") if os.path.exists(f"{OUT}/{f}"))
open(f"{OUT}/S7_report.txt", "w").write(rep or "!! no report produced"); print(rep)
json.dump(RC, open(f"{OUT}/return_codes.json", "w"), indent=1)
z = shutil.make_archive("/kaggle/working/S7_results", "zip", OUT)
print("ZIP:", z, f"({os.path.getsize(z) / 1e6:.1f} MB) | session {used_h():.2f} h | return codes {RC}")